# Embedding Baseline Models

Sanity-check pipeline:
1. Load precomputed DNABERT-2 embeddings (samples × 768)
2. Parse `RiceDiversity_44K_Phenotypes_34traits_PLINK.txt` → **StandardScaler'd numpy array**
3. Align samples between the two by NSFTVID
4. Fit quick baseline models (Ridge, Lasso, RF) on a few traits and report Pearson r

In [1]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

EMB_PATH      = Path("/home/adickson/rice_data/sativas413_embeddings.pt")
PHENO_PATH    = Path("../rice_data/RiceDiversity_44K_Phenotypes_34traits_PLINK.txt")

## 1. Load Embeddings
Load the precomputed DNABERT-2 embeddings.

In [2]:
data = torch.load(EMB_PATH, map_location="cpu", weights_only=True)
embeddings  = data["embeddings"].float().numpy()   # (413, 768)
emb_samples = data["sample_ids"]

# Center embeddings 
embeddings_centered = embeddings - embeddings.mean(axis=0)

print(f"Shape (samples × dims) : {embeddings.shape}")
print(f"First 5 sample IDs: {emb_samples[:5]}")

Shape (samples × dims) : (413, 768)
First 5 sample IDs: ['081215-A05_1', '081215-A06_3', '081215-A07_4', '081215-A08_5', '090414-A09_6']


## 2. Phenotype file → StandardScaler'd numpy array

In [3]:
pheno_raw = pd.read_csv(PHENO_PATH, sep="\t")

TRAIT_COLS = [c for c in pheno_raw.columns if c not in ("HybID", "NSFTVID")]
print(f"{len(TRAIT_COLS)} trait columns.")

# Extract NSFTVID from embedding sample names (suffix after last '_')
emb_nsftvid = [int(s.rsplit("_", 1)[-1]) for s in emb_samples]
emb_id_to_idx = {nid: i for i, nid in enumerate(emb_nsftvid)}

# Keep only phenotype rows that have a matching valid embedding sample
pheno_raw["NSFTVID"] = pheno_raw["NSFTVID"].astype(int)
pheno_matched = pheno_raw[pheno_raw["NSFTVID"].isin(emb_id_to_idx)].copy()
pheno_matched = pheno_matched.reset_index(drop=True)

print(f"Phenotype rows with matching embedding: {len(pheno_matched)} / {len(pheno_raw)}")

# Reorder embeddings rows to match phenotype order
emb_row_order = [emb_id_to_idx[nid] for nid in pheno_matched["NSFTVID"]]
X = embeddings_centered[emb_row_order, :]   # reordered matrix

print(f"Aligned X shape: {X.shape}")

# Build the scaled phenotype array
Y_raw = pheno_matched[TRAIT_COLS].values.astype(float)   # (n_samples, n_traits)
Y_scaled = np.full_like(Y_raw, np.nan)
for j in range(Y_raw.shape[1]):
    col = Y_raw[:, j]
    mask = ~np.isnan(col)
    if mask.sum() < 2:
        continue
    mu  = col[mask].mean()
    std = col[mask].std(ddof=0)
    Y_scaled[mask, j] = (col[mask] - mu) / (std if std > 0 else 1.0)

print(f"Y_scaled shape : {Y_scaled.shape}  (samples × traits)")


36 trait columns.
Phenotype rows with matching embedding: 413 / 413
Aligned X shape: (413, 768)
Y_scaled shape : (413, 36)  (samples × traits)


## 3. Dimensionality reduction (SVD)
Reduce dimensionality to match the baseline notebook pipeline.

In [4]:
from sklearn.decomposition import TruncatedSVD

# Reduce dimensionality with TruncatedSVD (120 components) to match baseline notebook
print("Fitting TruncatedSVD (120 components) on embeddings...")
svd = TruncatedSVD(n_components=120, random_state=42)
X_svd = svd.fit_transform(X)   # (n_samples, 120)
print(f"Explained variance ratio (first 10 components): "
      f"{svd.explained_variance_ratio_[:10].round(3)}")
print(f"Total variance explained: {svd.explained_variance_ratio_.sum():.3f}")

Fitting TruncatedSVD (120 components) on embeddings...
Explained variance ratio (first 10 components): [0.997 0.001 0.    0.    0.    0.    0.    0.    0.    0.   ]
Total variance explained: 1.000


## 4. Baseline models
Run Ridge, Lasso, and Random Forest on the traits using 5-fold CV. Reports Pearson r and R² on held-out folds.

In [5]:
from scipy.stats import pearsonr
from sklearn.linear_model import Ridge, Lasso, MultiTaskLasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

def cv_eval(X_feat, y, models, n_splits=5, random_state=42):
    """5-fold CV returning mean Pearson r and R² per model."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    results = {name: {"pearson_r": [], "r2": []} for name, _ in models}

    for train_idx, test_idx in kf.split(X_feat):
        X_tr, X_te = X_feat[train_idx], X_feat[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        for name, model in models:
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            r, _ = pearsonr(y_te, y_pred)
            results[name]["pearson_r"].append(r)
            results[name]["r2"].append(r2_score(y_te, y_pred))

    return {
        name: {
            "pearson_r": np.mean(vals["pearson_r"]),
            "r2":        np.mean(vals["r2"]),
            "n_splits": n_splits,
            "model": model,
            "r2_std": np.std(vals["r2"]),
            "pearson_r_std": np.std(vals["pearson_r"]),
        }
        for name, vals in results.items()
    }

MODELS = [
    ("Ridge",  Ridge(alpha=1.0)),
    ("Lasso",  Lasso(alpha=0.04, max_iter=5000)),
    ("RF-100", RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)),
    ("GBR-100", GradientBoostingRegressor(n_estimators=100, random_state=42)),
]

EVAL_TRAITS = TRAIT_COLS  # evaluate all traits

all_rows = []
for trait in EVAL_TRAITS:
    t_idx = TRAIT_COLS.index(trait)
    y_t   = Y_scaled[:, t_idx]
    mask  = ~np.isnan(y_t)
    if mask.sum() < 30:
        print(f"Skipping {trait!r} — too few non-NaN samples ({mask.sum()})")
        continue

    # Evaluate on the SVD features just like the baseline did with sparse ones
    res = cv_eval(X_svd[mask], y_t[mask], MODELS)
    for model_name, metrics in res.items():
        all_rows.append({
            "Trait": trait,
            "Model": model_name,
            "Pearson r": round(metrics["pearson_r"], 4) if metrics["pearson_r"] is not None else None,
            "R²":        round(metrics["r2"],        4) if metrics["r2"] is not None else None,
            "R² std":     round(metrics["r2_std"], 4) if metrics["r2_std"] is not None else None,
            "Pearson r std": round(metrics["pearson_r_std"], 4) if metrics["pearson_r_std"] is not None else None,
            "n_samples": int(mask.sum()),
        })

results_df = pd.DataFrame(all_rows)
results_df = results_df.sort_values(["Trait", "Pearson r"], ascending=[True, False])
results_df

/tmp/ipykernel_1245209/2622368471.py:19: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_te, y_pred)
/tmp/ipykernel_1245209/2622368471.py:19: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_te, y_pred)
/tmp/ipykernel_1245209/2622368471.py:19: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_te, y_pred)
/tmp/ipykernel_1245209/2622368471.py:19: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_te, y_pred)
/tmp/ipykernel_1245209/2622368471.py:19: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_te, y_pred)
/tmp/ipykernel_1245209/2622368471.py:19: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_te, y_pred

,Trait,Model,Pearson r,R²,R² std,Pearson r std,n_samples
130,Alkali spreading value,RF-100,0.4236,0.1660,0.0540,0.0744,403
131,Alkali spreading value,GBR-100,0.3968,0.1227,0.0369,0.0606,403
128,Alkali spreading value,Ridge,0.3341,-0.0156,0.0086,0.0729,403
129,Alkali spreading value,Lasso,NaN,-0.0165,0.0088,NaN,403
126,Amylose content,RF-100,0.6654,0.4311,0.0755,0.0520,401
...,...,...,...,...,...,...,...
141,Year06Flowering time at Arkansas,Lasso,NaN,-0.0173,0.0172,NaN,337
138,Year07Flowering time at Arkansas,RF-100,0.6034,0.3571,0.0712,0.0669,367
139,Year07Flowering time at Arkansas,GBR-100,0.5218,0.2626,0.0733,0.0605,367
136,Year07Flowering time at Arkansas,Ridge,0.4137,-0.0076,0.0098,0.1051,367


In [6]:
for trait, subdf in results_df[["Trait", "Model", "Pearson r", "R²", "R² std"]].groupby("Trait"):
    print(f"\nTrait: {trait}")
    print(subdf[["Model", "Pearson r", "R²", "R² std"]].to_string(index=False))


Trait: Alkali spreading value
  Model  Pearson r      R²  R² std
 RF-100     0.4236  0.1660  0.0540
GBR-100     0.3968  0.1227  0.0369
  Ridge     0.3341 -0.0156  0.0086
  Lasso        NaN -0.0165  0.0088

Trait: Amylose content
  Model  Pearson r      R²  R² std
 RF-100     0.6654  0.4311  0.0755
GBR-100     0.6404  0.3805  0.1183
  Ridge     0.5748 -0.0063  0.0075
  Lasso        NaN -0.0087  0.0080

Trait: Awn presence
  Model  Pearson r      R²  R² std
GBR-100     0.2488  0.0003  0.1362
 RF-100     0.2140  0.0266  0.0759
  Ridge     0.0912 -0.0087  0.0110
  Lasso        NaN -0.0087  0.0110

Trait: Blast resistance
  Model  Pearson r      R²  R² std
 RF-100     0.5626  0.2995  0.0944
GBR-100     0.5289  0.2571  0.0679
  Ridge     0.4485 -0.0233  0.0251
  Lasso        NaN -0.0248  0.0252

Trait: Brown rice length/width ratio
  Model  Pearson r      R²  R² std
 RF-100     0.6656  0.4197  0.0446
GBR-100     0.6440  0.3879  0.0358
  Ridge     0.4689 -0.0252  0.0305
  Lasso        NaN -0

In [ ]:
# Heatmap of Pearson r across traits × models
pivot = results_df.pivot(index="Trait", columns="Model", values="Pearson r")
pivot = pivot.dropna(how='all')

fig_height = max(4, int(len(pivot.index) * 0.25))
fig, ax = plt.subplots(figsize=(8, fig_height))
im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=-0.2, vmax=1.0)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=10)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_title("5-fold CV Pearson r (Embedding PCA features, 120 components)")
plt.colorbar(im, ax=ax, label="Pearson r")

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=9)

plt.tight_layout()
plt.show()